# Homework 10
#### Course Notes
**Language Models:** https://github.com/rjenki/BIOS512/tree/main/lecture17  
**Unix:** https://github.com/rjenki/BIOS512/tree/main/lecture18  
**Docker:** https://github.com/rjenki/BIOS512/tree/main/lecture19

## Question 1
#### Make a language model that uses ngrams and allows the user to specify start words, but uses a random start if one is not specified.

#### a) Make a function to tokenize the text.

In [20]:
library(httr)
library(tokenizers)
library(stringr)

In [21]:
tokenize <- function(text) {
  tokenizers::tokenize_words(text, lowercase = TRUE, strip_punct = TRUE)[[1]]
}

#### b) Make a function generate keys for ngrams.

In [35]:
gen_keys <- function(tokens, n) {
  pairs <- list()
  if (n < 1) stop("n must be >= 1")
  if (n == 1) {
    for (i in seq_along(tokens)) {
      pairs[[length(pairs) + 1]] <- list(ctx = character(0), next_word = tokens[i])
    }
    return(pairs)
  }
  if (length(tokens) < n) return(pairs)
  k <- 1
  for (i in seq_len(length(tokens) - n + 1)) {
    ctx <- tokens[i:(i + n - 2)]     # n-1 tokens
    nxt <- tokens[i + n - 1]         # the next token
    pairs[[k]] <- list(ctx = ctx, next_word = nxt)
    k <- k + 1
  }
  pairs
}

#### c) Make a function to build an ngram table.

In [36]:
build_ngram_model <- function(tokens, n) {
  model <- list()
  pairs <- gen_keys(tokens, n)
  for (p in pairs) {
    key <- if (length(p$ctx) == 0) "" else paste(p$ctx, collapse = " ")
    if (is.null(model[[key]])) model[[key]] <- integer(0)
    counts <- model[[key]]
    
    nw <- p$next_word
    if (is.na(counts[nw])) counts[nw] <- 0L
    counts[nw] <- counts[nw] + 1L
    model[[key]] <- counts
  }
  model
}

#### d) Function to digest the text.

In [37]:
library(tokenizers)

digest_text <- function(text) {
  # 1. Clean up any extra spaces or line breaks
  text <- gsub("\\s+", " ", text)

  # 2. Turn the text into lowercase words (no punctuation)
  tokens <- tokenizers::tokenize_words(text, lowercase = TRUE, strip_punct = TRUE)[[1]]

  # 3. Give back the list of words
  tokens
}

#### e) Function to digest the url.

In [38]:
digest_url <- function(url) {
  resp <- httr::GET(url)
  if (httr::http_error(resp)) stop("Could not download the URL.")
  body <- httr::content(resp, as = "text", encoding = "UTF-8")
  text_plain <- gsub("<[^>]+>", " ", body)
  text_plain <- gsub("\\s+", " ", text_plain)
  tokenize(text_plain)
}

#### f) Function that gives random start.

In [39]:
random_start <- function(model, n) {
  if (n == 1) return(character(0))
  keys <- names(model)
 
  keys_shuf <- sample(keys)
  for (k in keys_shuf) {
    toks <- if (k == "") character(0) else strsplit(k, " ")[[1]]
    if (length(toks) == n - 1) return(toks)
  }

  first <- if (length(keys) > 0) keys[1] else ""
  toks <- if (first == "") character(0) else strsplit(first, " ")[[1]]
  if (length(toks) >= n - 1) tail(toks, n - 1) else c(rep("<s>", (n-1)-length(toks)), toks)
}

#### g) Function to predict the next word.

In [40]:
predict_next_word <- function(model, context, deterministic = FALSE) {
  key <- if (length(context) == 0) "" else paste(context, collapse = " ")
  counts <- model[[key]]
  if (is.null(counts) || length(counts) == 0) {
 
    counts <- model[[ sample(names(model), 1) ]]
  }
  words <- names(counts)
  freqs <- as.numeric(counts)
  if (deterministic) {
    words[which.max(freqs)]
  } else {
    probs <- freqs / sum(freqs)
    sample(words, size = 1, prob = probs)
  }
}

#### h) Function that puts everything together. Specify that if the user does not give a start word, then the random start will be used.

In [41]:
build_and_generate <- function(text_or_url, n = 3, length = 50, start = NULL, deterministic = FALSE, is_url = FALSE) {
  tokens <- if (is_url) digest_url(text_or_url) else tokenize(text_or_url)
  model <- build_ngram_model(tokens, n)
  
  if (n == 1) context <- character(0) else {
    if (!is.null(start)) {
      st_toks <- tokenize(start)
      if (length(st_toks) >= n-1) context <- tail(st_toks, n-1) else context <- c(rep("<s>", (n-1)-length(st_toks)), st_toks)
    } else {
      context <- random_start(model, n)
    }
  }
  
  out <- if (n == 1) character(0) else as.character(context)
  for (i in seq_len(length)) {
    nxt <- predict_next_word(model, context, deterministic = deterministic)
    out <- c(out, nxt)
    if (n == 1) context <- character(0) else context <- tail(c(context, nxt), n-1)
  }
  paste(out, collapse = " ")
}

## Question 2
#### For this question, set `seed=2025`.
#### a) Test your model using a text file of [Grimm's Fairy Tails](https://www.gutenberg.org/cache/epub/2591/pg2591.txt)
#### i) Using n=3, with the start word(s) "the king", with length=15. 
#### ii) Using n=3, with no start word, with length=15.

In [46]:
gen_from_model <- function(model, n = 3, start = NULL, length_out = 15) {

  if (is.null(start)) {
    key <- sample(names(model), 1)                      # pick an existing context
    context <- if (nzchar(key)) strsplit(key, " ")[[1]] else character(0)
  } else {
    st <- tokenizers::tokenize_words(start, lowercase = TRUE, strip_punct = TRUE)[[1]]
    context <- if (length(st) >= n-1) tail(st, n-1) else c(rep("<s>", (n-1)-length(st)), st)
  }
  out <- context
  for (i in seq_len(length_out)) {
    key <- if (length(context) == 0) "" else paste(context, collapse = " ")
    counts <- model[[key]]
    if (is.null(counts) || length(counts) == 0) {
     
      nonempty <- names(model)[sapply(names(model), function(k) length(model[[k]]) > 0)]
      counts <- model[[ sample(nonempty, 1) ]]
    }
    words <- names(counts); freqs <- as.numeric(counts)
    nxt <- sample(words, size = 1, prob = freqs / sum(freqs))
    out <- c(out, nxt)
    context <- tail(c(context, nxt), n - 1)
  }
  paste(out, collapse = " ")
}

set.seed(2025)
out_ia  <- gen_from_model(model_grimm, n = 3, start = "the king", length_out = 15)
set.seed(2025)
out_iia <- gen_from_model(model_grimm, n = 3, start = NULL,        length_out = 15)

cat("Grimm's Fairy Tales (start = 'the king'):\n", out_ia, "\n\n")
cat("Grimm's Fairy Tales (random start):\n", out_iia, "\n")

Grimm's Fairy Tales (start = 'the king'):
 the king has forbidden me to marry another husband am not i shall ride upon this fear 

Grimm's Fairy Tales (random start):
 die he begged for their prey but if i am to live upon and the gardener so 


#### b) Test your model using a text file of [Ancient Armour and Weapons in Europe](https://www.gutenberg.org/cache/epub/46342/pg46342.txt)
#### i) Using n=3, with the start word(s) "the king", with length=15. 
#### ii) Using n=3, with no start word, with length=15.

In [47]:
set.seed(2025)

ancient_url <- "https://www.gutenberg.org/cache/epub/46342/pg46342.txt"
tokens_ancient <- digest_url(ancient_url)

model_ancient <- build_ngram_model(tokens_ancient, 3)

# (i)
if ("the king" %in% names(model_ancient)) {
  out_ib <- gen_from_model(model_ancient, n = 3, start = "the king", length_out = 15)
} else {
  king_keys <- names(model_ancient)[grepl("\\bking\\b", names(model_ancient))]
  if (length(king_keys) > 0) {
    out_ib <- gen_from_model(model_ancient, n = 3, start = king_keys[1], length_out = 15)
  } else {
    warning("No 'king' context exists in model — using random start for out_i.")
    out_ib <- gen_from_model(model_ancient, n = 3, start = NULL, length_out = 15)
  }
}

# (ii) random start
out_iib <- gen_from_model(model_ancient, n = 3, start = NULL, length_out = 15)

cat("Ancient Armour & Weapons (start = 'the king' or fallback):\n", out_ib, "\n\n")
cat("Ancient Armour & Weapons (random start):\n", out_iib, "\n")

Ancient Armour & Weapons (start = 'the king' or fallback):
 the king he added to the entire exclusion of the swords were made prisoners the saxon chronicle 

Ancient Armour & Weapons (random start):
 kinds even to the number of eighty were hanged the archers with arrows or with other weapons 


#### c) Explain in 1-2 sentences the difference in content generated from each source.

In [ ]:
The Grimm model produces storytelling style phrases (family, emotion, dialogue), 
whereas the Ancient Armour model produces technical, historical phrases because
trigram models reproduce the short-word patterns from their training texts.

## Question 3
#### a) What is a language learning model? 
#### b) Imagine the internet goes down and you can't run to your favorite language model for help. How do you run one locally?

In [ ]:
a. A large language model (LLM) is a type of artificial intelligence (AI) program that can recognize and generate text, among other tasks. 
LLMs are trained on huge sets of data and are built on machine learning: a type of neural network called a transformer model

In [ ]:
b. To run a large language model (LLM) on your computer, you need to download and install software like LM Studio or Ollama along with a model file. 
Then, you can load and run the model on your device to use it offline for things like chatting or coding without needing an internet connection.

## Question 4
#### Explain what the following vocab words mean in the context of typing `mkdir project` into the command line. If the term doesn't apply to this command, give the definition and/or an example.
| Term | Meaning |  
|------|---------|
| **Shell** | A program that acts as an interface between a user and the operating system, allowing us to interact with the computer by typing commands or using a graphical interface. Eg Bash, Zsh |
| **Terminal emulator** | Window or app you type the command into. It lets you interact with the shell. Eg macOS Terminal, Windows PowerShell |
| **Process** | A series of steps and decisions to achieve a goal, including steps, decisions, variability, timing, and resource assignment. When we run mkdir project, the shell starts a process, which is a running instance of the mkdir program that creates the folder. |
| **Signal** | A message sent to a process to control it. Eg pressing Ctrl + C sends a signal to stop a running process. |
| **Standard input** | Data the program can read from the keyboard or another program.|
| **Standard output** | Standard output is where a program sends its normal output text. |
| **Command line argument** | A command line argument is what we type after the command. In mkdir project, the word project is the argument, it tells mkdir what folder to make.  |
| **The environment** | TThe set of settings and variables the shell gives to every process. Eg it includes info like your username or current directory. mkdir can use this info to know where to create the folder. |

## Question 5
#### Consider the following command `find . -iname "*.R" | xargs grep read_csv`.
#### a) What are the programs?
#### b) Explain what this command is doing, part by part.

In [ ]:
a. The command uses three programs:
find – searches for files and directories.
xargs – takes input from another command and passes it as arguments to another command.
grep – For each R file found, it looks inside it for the word read_csv

In [ ]:
b. This command searches all R script files in the current directory and its subdirectories for any line that contains the text read_csv.

find - Tells find to start looking in the current directory (the . means "here").
-iname "*.R" - Looks for all files ending in .R (R script files), ignoring case (so it matches .r or .R).
| - The pipe symbol means send the results to the next program.
xargs grep read_csv - For each file name that find outputs, xargs runs grep read_csv — meaning it searches inside each R file for any line containing the text read_csv.

## Question 6
#### Install Docker on your machine. See [here](https://github.com/rjenki/BIOS512/blob/main/lecture18/docker_install.md) for instructions. 
#### a) Show the response when you run `docker run hello-world`.
#### b) Access Rstudio through a Docker container. Set your password and make sure your files show up on the Rstudio server. Type the command and the output you get below.
#### c) How do you log in to the RStudio server?

In [ ]:
a. Hello from Docker!
This message shows that your installation appears to be working correctly.

In [ ]:
b.
#Command
mkdir -p ~/rstudio
ls -ld ~/rstudio

docker run -d --name my-rstudio -p 8787:8787 -v /Users/shila/rstudio:/home/rstudio -e 'PASSWORD=123#Shila' rocker/rstudio

#Output
Unable to find image 'rocker/rstudio:latest' locally
latest: Pulling from rocker/rstudio
3665120d345d: Pull complete 
2034506aa72f: Pull complete 
b8a35db46e38: Pull complete 
bcce866b1806: Pull complete 
664fb1818bbb: Pull complete 
91ed5b86de88: Pull complete 
191985778909: Pull complete 
cc9c938c1f51: Pull complete 
e2804bef35e8: Pull complete 
5d246ec925db: Pull complete 
5b219f62ce36: Pull complete 
08e74fd5985d: Pull complete 
abd0190d83fb: Pull complete 
2c9ba66d5dbe: Pull complete 
bc9245ceaac5: Pull complete 
9c1a4a0706b7: Pull complete 
a730ff463d58: Pull complete 
39038e16d1ba: Pull complete 
Digest: sha256:9f85211a666fb426081a6f5a01f9f9f51655262258419fa21e0ce38a5afc78d8
Status: Downloaded newer image for rocker/rstudio:latest
5aeba10d8ca530a4446acd44ec28b337148d8db90855fb25c14739057716c1f3

In [ ]:
c. Go to webbrowser and paste and run the following link and login using the login info set up before.
http://localhost:8787